# Importing Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, MaxPooling2D, Flatten, Dense
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Dropout
from tensorflow.keras.layers import concatenate, Activation

from tensorflow.keras import Model

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc


from imblearn.metrics import specificity_score


from  matplotlib import pyplot as plt
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True



# Input data

In [ ]:
targetnames = ['No_DR', 'Mild_DR', 'Moderate_DR', 'Severe_DR', 'Proliferative_DR']
classnames = ['No_DR', 'Mild_DR', 'Moderate_DR', 'Severe_DR', 'Proliferative_DR']
num_classes = 5
image_size = 224
batch_size = 16

# QS calculation

In [ ]:
import numpy as np

def compute_q_statistic(correct_i, correct_j):

    N11 = np.sum(correct_i & correct_j)
    N00 = np.sum(~correct_i & ~correct_j)
    N01 = np.sum(~correct_i & correct_j)
    N10 = np.sum(correct_i & ~correct_j)
    numerator = N11 * N00 - N01 * N10
    denominator = N11 * N00 + N01 * N10
    return numerator / denominator if denominator != 0 else 0

# ISA Models' Predictions

## DN

### DN121

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_DenseNet121_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_DN121 (predicted classes)
WTavg_predictions_wt_DN121 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_DN121 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_DN121 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_DN121 shape:", WTavg_predictions_wt_DN121.shape)
# print("weighted_predictions_DN121 shape:", weighted_predictions_DN121.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_DN121
y_prob = weighted_predictions_DN121
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_DN121.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_DN121.pdf", format='pdf', bbox_inches='tight')

plt.show()

### DN169

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_DenseNet169_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_DN169 (predicted classes)
WTavg_predictions_wt_DN169 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_DN169 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_DN169 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_DN169 shape:", WTavg_predictions_wt_DN169.shape)
# print("weighted_predictions_DN169 shape:", weighted_predictions_DN169.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_DN169
y_prob = weighted_predictions_DN169
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_DN169.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_DN169.pdf", format='pdf', bbox_inches='tight')

plt.show()

### DN201

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_DenseNet201_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_DN201 (predicted classes)
WTavg_predictions_wt_DN201 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_DN201 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_DN201 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_DN201 shape:", WTavg_predictions_wt_DN201.shape)
# print("weighted_predictions_DN201 shape:", weighted_predictions_DN201.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_DN201
y_prob = weighted_predictions_DN201
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_DN201.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_DN201.pdf", format='pdf', bbox_inches='tight')

plt.show()

### IA_DN

In [ ]:
# List of model predictions (assuming these are probability vectors)
model_predictions = [
    weighted_predictions_DN121,
    weighted_predictions_DN169,
    weighted_predictions_DN201
]

# True labels
labels = y_true

# Convert model predictions to class labels
hard_predictions = [np.argmax(pred, axis=1) for pred in model_predictions]

# Calculate correctness for each model (boolean array)
correctness = [pred == labels for pred in hard_predictions]
n_models = len(model_predictions)

# Compute pairwise Q-Statistics
Q_matrix = np.zeros((n_models, n_models))
for i in range(n_models):
    for j in range(i+1, n_models):
        q_ij = compute_q_statistic(correctness[i], correctness[j])
        Q_matrix[i, j] = q_ij
        Q_matrix[j, i] = q_ij

# Calculate average Q-Statistic per model
avg_Q = (np.sum(Q_matrix, axis=1) / (n_models - 1))

# Convert Q to weights (lower Q = more diverse = higher weight)
weights = 1 - avg_Q  # Q ranges [-1,1] → weights [0,2]
weights = np.maximum(weights, 0)  # Ensure non-negative
weights /= np.sum(weights)  # Normalize

# Compute weighted ensemble predictions
weighted_predictions_ia_DN = np.sum([w * pred for w, pred in zip(weights, model_predictions)], axis=0)
final_predictions = np.argmax(weighted_predictions_ia_DN, axis=1)

# Evaluate accuracy
accuracy = np.mean(final_predictions == labels)
print("Q-Statistic Ensemble Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_IA_DN.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_IA_DN.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd

# Create a DataFrame with predictions and probabilities
predictions_df = pd.DataFrame({
    "True_Labels": y_true,
    "Predicted_Labels": y_pred,
})

# Add probabilities if needed (optional)
for i in range(y_prob.shape[1]):
    predictions_df[f"Prob_Class_{i}"] = y_prob[:, i]

# Save to CSV
output_file = "predictions_IA_DN.csv"
predictions_df.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")


## RN

### RN50

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_ResNet50_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_RN50 (predicted classes)
WTavg_predictions_wt_RN50 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_RN50 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_RN50 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_RN50 shape:", WTavg_predictions_wt_RN50.shape)
# print("weighted_predictions_RN50 shape:", weighted_predictions_RN50.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_RN50
y_prob = weighted_predictions_RN50
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_RN50.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_RN50.pdf", format='pdf', bbox_inches='tight')

plt.show()

### RN101

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_ResNet101_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_RN101 (predicted classes)
WTavg_predictions_wt_RN101 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_RN101 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_RN101 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_RN101 shape:", WTavg_predictions_wt_RN101.shape)
# print("weighted_predictions_RN101 shape:", weighted_predictions_RN101.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_RN101
y_prob = weighted_predictions_RN101
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_RN101.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_RN101.pdf", format='pdf', bbox_inches='tight')

plt.show()

### RN152

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_DenseNet201_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_RN152 (predicted classes)
WTavg_predictions_wt_RN152 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_RN152 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_RN152 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_RN152 shape:", WTavg_predictions_wt_RN152.shape)
# print("weighted_predictions_RN152 shape:", weighted_predictions_RN152.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_RN152
y_prob = weighted_predictions_RN152
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_RN152.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_RN152.pdf", format='pdf', bbox_inches='tight')

plt.show()

### IA_RN

In [ ]:
# List of model predictions (assuming these are probability vectors)
model_predictions = [
    weighted_predictions_RN50,
    weighted_predictions_RN101,
    weighted_predictions_RN152
]

# True labels
labels = y_true

# Convert model predictions to class labels
hard_predictions = [np.argmax(pred, axis=1) for pred in model_predictions]

# Calculate correctness for each model (boolean array)
correctness = [pred == labels for pred in hard_predictions]
n_models = len(model_predictions)

# Compute pairwise Q-Statistics
Q_matrix = np.zeros((n_models, n_models))
for i in range(n_models):
    for j in range(i+1, n_models):
        q_ij = compute_q_statistic(correctness[i], correctness[j])
        Q_matrix[i, j] = q_ij
        Q_matrix[j, i] = q_ij

# Calculate average Q-Statistic per model
avg_Q = (np.sum(Q_matrix, axis=1) / (n_models - 1))

# Convert Q to weights (lower Q = more diverse = higher weight)
weights = 1 - avg_Q  # Q ranges [-1,1] → weights [0,2]
weights = np.maximum(weights, 0)  # Ensure non-negative
weights /= np.sum(weights)  # Normalize

# Compute weighted ensemble predictions
weighted_predictions_ia_RN = np.sum([w * pred for w, pred in zip(weights, model_predictions)], axis=0)
final_predictions = np.argmax(weighted_predictions_ia_RN, axis=1)

# Evaluate accuracy
accuracy = np.mean(final_predictions == labels)
print("Q-Statistic Ensemble Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_IA_RN.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_IA_RN.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd

# Create a DataFrame with predictions and probabilities
predictions_df = pd.DataFrame({
    "True_Labels": y_true,
    "Predicted_Labels": y_pred,
})

# Add probabilities if needed (optional)
for i in range(y_prob.shape[1]):
    predictions_df[f"Prob_Class_{i}"] = y_prob[:, i]

# Save to CSV
output_file = "predictions_IA_RN.csv"
predictions_df.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")


## Inception

### Iv3

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_Inception_V3_aptos2019_2.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_Iv3 (predicted classes)
WTavg_predictions_wt_Iv3 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_Iv3 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_Iv3 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_Iv3 shape:", WTavg_predictions_wt_Iv3.shape)
# print("weighted_predictions_Iv3 shape:", weighted_predictions_Iv3.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_Iv3
y_prob = weighted_predictions_Iv3
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_Iv3.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_Iv3.pdf", format='pdf', bbox_inches='tight')

plt.show()

### IRv2

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_Inception_Resnet_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_IRv2 (predicted classes)
WTavg_predictions_wt_IRv2 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_IRv2 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_IRv2 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_IRv2 shape:", WTavg_predictions_wt_IRv2.shape)
# print("weighted_predictions_IRv2 shape:", weighted_predictions_IRv2.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_IRv2
y_prob = weighted_predictions_IRv2
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_IRv2.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_IRv2.pdf", format='pdf', bbox_inches='tight')

plt.show()

### X

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_Xception_aptos2019_2.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_X (predicted classes)
WTavg_predictions_wt_X = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_X (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_X = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_X shape:", WTavg_predictions_wt_X.shape)
# print("weighted_predictions_X shape:", weighted_predictions_X.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_X
y_prob = weighted_predictions_X
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_X.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_X.pdf", format='pdf', bbox_inches='tight')

plt.show()

### IA_Inception

In [ ]:
# List of model predictions (assuming these are probability vectors)
model_predictions = [
    weighted_predictions_Iv3,
    weighted_predictions_IRv2,
    weighted_predictions_X
]

# True labels
labels = y_true

# Convert model predictions to class labels
hard_predictions = [np.argmax(pred, axis=1) for pred in model_predictions]

# Calculate correctness for each model (boolean array)
correctness = [pred == labels for pred in hard_predictions]
n_models = len(model_predictions)

# Compute pairwise Q-Statistics
Q_matrix = np.zeros((n_models, n_models))
for i in range(n_models):
    for j in range(i+1, n_models):
        q_ij = compute_q_statistic(correctness[i], correctness[j])
        Q_matrix[i, j] = q_ij
        Q_matrix[j, i] = q_ij

# Calculate average Q-Statistic per model
avg_Q = (np.sum(Q_matrix, axis=1) / (n_models - 1))

# Convert Q to weights (lower Q = more diverse = higher weight)
weights = 1 - avg_Q  # Q ranges [-1,1] → weights [0,2]
weights = np.maximum(weights, 0)  # Ensure non-negative
weights /= np.sum(weights)  # Normalize

# Compute weighted ensemble predictions
weighted_predictions_ia_Inception = np.sum([w * pred for w, pred in zip(weights, model_predictions)], axis=0)
final_predictions = np.argmax(weighted_predictions_ia_Inception, axis=1)

# Evaluate accuracy
accuracy = np.mean(final_predictions == labels)
print("Q-Statistic Ensemble Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_IA_Inception.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_IA_Inception.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd

# Create a DataFrame with predictions and probabilities
predictions_df = pd.DataFrame({
    "True_Labels": y_true,
    "Predicted_Labels": y_pred,
})

# Add probabilities if needed (optional)
for i in range(y_prob.shape[1]):
    predictions_df[f"Prob_Class_{i}"] = y_prob[:, i]

# Save to CSV
output_file = "predictions_IA_Inception.csv"
predictions_df.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")


## MN

### MNv1

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_MobileNET_V1_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_MNv1 (predicted classes)
WTavg_predictions_wt_MNv1 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_MNv1 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_MNv1 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_MNv1 shape:", WTavg_predictions_wt_MNv1.shape)
# print("weighted_predictions_MNv1 shape:", weighted_predictions_MNv1.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_MNv1
y_prob = weighted_predictions_MNv1
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_MNv1.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_MNv1.pdf", format='pdf', bbox_inches='tight')

plt.show()

### MNv2

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_MobileNET_V2_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_MNv2 (predicted classes)
WTavg_predictions_wt_MNv2 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_MNv2 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_MNv2 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_MNv2 shape:", WTavg_predictions_wt_MNv2.shape)
# print("weighted_predictions_MNv2 shape:", weighted_predictions_MNv2.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_MNv2
y_prob = weighted_predictions_MNv2
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_MNv2.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_MNv2.pdf", format='pdf', bbox_inches='tight')

plt.show()

### MNv3

In [ ]:
# Path to the saved file
file_path = "/content/drive/MyDrive/Research/IQSA/predictions_MobileNET_V3_aptos2019.csv"

# Load the predictions file
predictions_df = pd.read_csv(file_path)

# Extract the WTavg_predictions_wt_MNv3 (predicted classes)
WTavg_predictions_wt_MNv3 = predictions_df["Predicted_Labels"].to_numpy()

# Extract the weighted_predictions_MNv3 (probabilities)
# Assuming probabilities are stored in columns "Prob_Class_0", "Prob_Class_1", ...
probability_columns = [col for col in predictions_df.columns if col.startswith("Prob_Class")]
weighted_predictions_MNv3 = predictions_df[probability_columns].to_numpy()

# Check the shapes to ensure everything is correct
# print("WTavg_predictions_wt_MNv3 shape:", WTavg_predictions_wt_MNv3.shape)
# print("weighted_predictions_MNv3 shape:", weighted_predictions_MNv3.shape)

# Use these as needed
y_pred = WTavg_predictions_wt_MNv3
y_prob = weighted_predictions_MNv3
y_true = predictions_df["True_Labels"].to_numpy()

# Evaluate the accuracy of the weighted average predictions on the test data
accuracy = np.mean(y_pred == y_true)
print("Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_MNv3.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
### Individual Class Accuracy
# y_true are the true labels of the test set
# y_pred are the predicted labels of the test set
accuracy = accuracy_score(y_true, y_pred)
# Calculate accuracy for each class
class_accuracy = cm.diagonal()/cm.sum(axis=1)
print("Overall accuracy: {:.2f}%".format(accuracy*100))
print("Accuracy for each class:")
for i, acc in enumerate(class_accuracy):
    print("Class {}: {:.2f}%".format(i, acc*100))

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_MNv3.pdf", format='pdf', bbox_inches='tight')

plt.show()

### IA_MN

In [ ]:
# List of model predictions (assuming these are probability vectors)
model_predictions = [
    weighted_predictions_MNv1,
    weighted_predictions_MNv2,
    weighted_predictions_MNv3
]

# True labels
labels = y_true

# Convert model predictions to class labels
hard_predictions = [np.argmax(pred, axis=1) for pred in model_predictions]

# Calculate correctness for each model (boolean array)
correctness = [pred == labels for pred in hard_predictions]
n_models = len(model_predictions)

# Compute pairwise Q-Statistics
Q_matrix = np.zeros((n_models, n_models))
for i in range(n_models):
    for j in range(i+1, n_models):
        q_ij = compute_q_statistic(correctness[i], correctness[j])
        Q_matrix[i, j] = q_ij
        Q_matrix[j, i] = q_ij

# Calculate average Q-Statistic per model
avg_Q = (np.sum(Q_matrix, axis=1) / (n_models - 1))

# Convert Q to weights (lower Q = more diverse = higher weight)
weights = 1 - avg_Q  # Q ranges [-1,1] → weights [0,2]
weights = np.maximum(weights, 0)  # Ensure non-negative
weights /= np.sum(weights)  # Normalize

# Compute weighted ensemble predictions
weighted_predictions_ia_MN = np.sum([w * pred for w, pred in zip(weights, model_predictions)], axis=0)
final_predictions = np.argmax(weighted_predictions_ia_MN, axis=1)

# Evaluate accuracy
accuracy = np.mean(final_predictions == labels)
print("Q-Statistic Ensemble Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_IA_MN.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_IA_MN.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd

# Create a DataFrame with predictions and probabilities
predictions_df = pd.DataFrame({
    "True_Labels": y_true,
    "Predicted_Labels": y_pred,
})

# Add probabilities if needed (optional)
for i in range(y_prob.shape[1]):
    predictions_df[f"Prob_Class_{i}"] = y_prob[:, i]

# Save to CSV
output_file = "predictions_IA_MN.csv"
predictions_df.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")


# IQSA

In [ ]:
# List of model predictions (assuming these are probability vectors)
model_predictions = [
    weighted_predictions_ia_DN,
    weighted_predictions_ia_RN,
    weighted_predictions_ia_Inception,
    weighted_predictions_ia_MN
]

# True labels
labels = y_true

# Convert model predictions to class labels
hard_predictions = [np.argmax(pred, axis=1) for pred in model_predictions]

# Calculate correctness for each model (boolean array)
correctness = [pred == labels for pred in hard_predictions]
n_models = len(model_predictions)

# Compute pairwise Q-Statistics
Q_matrix = np.zeros((n_models, n_models))
for i in range(n_models):
    for j in range(i+1, n_models):
        q_ij = compute_q_statistic(correctness[i], correctness[j])
        Q_matrix[i, j] = q_ij
        Q_matrix[j, i] = q_ij

# Calculate average Q-Statistic per model
avg_Q = (np.sum(Q_matrix, axis=1) / (n_models - 1))

# Convert Q to weights (lower Q = more diverse = higher weight)
weights = 1 - avg_Q  # Q ranges [-1,1] → weights [0,2]
weights = np.maximum(weights, 0)  # Ensure non-negative
weights /= np.sum(weights)  # Normalize

# Compute weighted ensemble predictions
weighted_predictions_ia_MN = np.sum([w * pred for w, pred in zip(weights, model_predictions)], axis=0)
final_predictions = np.argmax(weighted_predictions_ia_MN, axis=1)

# Evaluate accuracy
accuracy = np.mean(final_predictions == labels)
print("Q-Statistic Ensemble Accuracy:", accuracy)

In [ ]:
### Weighted Evaluation Measures
print("Weighted Precision: "+ str(precision_score(y_true, y_pred, average='weighted')))
print("Weighted Recall (sensitivity): "+ str(recall_score(y_true, y_pred, average='weighted')))
print("Weighted F1-Score: "+ str(f1_score(y_true, y_pred, average='weighted')))
print("Weighted Specificity: "+ str(specificity_score(y_true, y_pred, average='weighted')))

In [ ]:
## Confusion Matrix
title_font = {'fontname':'Times New Roman', 'size':'14', 'color':'black', 'weight':'normal'}
label_font = {'fontname':'Times New Roman', 'size':'12', 'color':'black', 'weight':'normal'}

from sklearn.metrics import confusion_matrix
import seaborn as sns
# Creating confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Plotting the confusion matrix
plt.figure(figsize=(5,3))
sns.heatmap(cm, annot=True, fmt='g', cmap='Greens', xticklabels=classnames, yticklabels=classnames)
plt.xlabel('Predicted Label', fontdict=label_font)
plt.ylabel('True Label', fontdict=label_font)
#plt.title('Confusion Matrix', fontdict=title_font)

# Adjust layout to minimize extra spaces
plt.tight_layout()
# Save the plot as a PDF file with minimized extra spaces
plt.savefig("confusion_matrix_IQSA.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
## ROC
label_font = {'size': 12, 'family': 'Times New Roman'}

# Import roc_curve
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true == i, y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Define the figure size
figsize = (4, 3)

# plot the ROC curve for each class
plt.figure(figsize=figsize)
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label='Class {}: AUC={:.3f}'.format(classnames[i], roc_auc[i]))

plt.xlabel('False Positive Rate', fontdict=label_font)
plt.ylabel('True Positive Rate', fontdict=label_font)
plt.legend()

# Adjust layout to minimize extra spaces
plt.tight_layout()

# Save the plot as a PDF file with minimized extra spaces
plt.savefig("roc_IQSA.pdf", format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd

# Create a DataFrame with predictions and probabilities
predictions_df = pd.DataFrame({
    "True_Labels": y_true,
    "Predicted_Labels": y_pred,
})

# Add probabilities if needed (optional)
for i in range(y_prob.shape[1]):
    predictions_df[f"Prob_Class_{i}"] = y_prob[:, i]

# Save to CSV
output_file = "predictions_IQSA.csv"
predictions_df.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")


In [ ]:

import os

filename = "predictions_IA_Inception.csv"

print("Current directory:", os.getcwd())
print("File exists:", os.path.exists(filename))
